In [20]:
import os
import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv

from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder


# Modelo de Atribución Probabilística de Impacto Ofensivo

## 1. Objetivo del Modelo
El propósito de este modelo es determinar la **distribución de probabilidad de participación en acciones ofensivas** (goles y asistencias) dentro del equipo. 

En lugar de predecir valores absolutos futuros a partir de un volumen reducido de datos, el modelo responde a preguntas tácticas clave:
* *Cuando el equipo marca un gol, ¿cuál es la probabilidad de que haya sido generado por cada línea posicional (Delanteros, Centrocampistas, Laterales, etc.)?*
* *¿Qué cuota de responsabilidad e impacto individual asume cada jugador en la producción ofensiva total del club?*

## 2. Justificación de la Metodología (¿Por qué un Modelo Probabilístico?)
Para datasets de tamaño moderado o agregados por temporada (como una plantilla de 19 jugadores), los modelos supervisados tradicionales de regresión o árboles complejos tienden al **sobreajuste (overfitting)** o a generar métricas engañosas.

Por ello, se ha optado por un enfoque de **Modelado de Atribución basado en Naive Bayes Multinomial**:
* **Alineación con datos de frecuencia:** Se adapta perfectamente a la naturaleza discreta de los eventos (goles y asistencias).
* **Robustez estadística:** Permite transformar métricas acumuladas en probabilidades a priori y a posteriori sin asumir relaciones lineales forzadas.
* **Valor para el cuerpo técnico:** Ofrece una interpretación táctica directa sobre la dependencia que tiene el equipo de determinadas posiciones o nombres propios a la hora de atacar.

## 3. Arquitectura del Flujo de Trabajo
1. **Extracción y Carga:** Conexión a la base de datos PostgreSQL para recuperar las métricas base por jugador.
2. **Ingeniería de Eventos (Resampling):** Desagregación de las estadísticas acumuladas para construir un dataset de eventos individuales de gol y asistencia.
3. **Entrenamiento (Naive Bayes):** Ajuste del clasificador probabilístico para estimar $P(\text{Posición} \mid \text{Tipo de Evento})$.
4. **Matriz de Impacto Individual:** Cálculo de la cuota relativa de contribución directa por jugador sobre el total colectivo.

In [21]:
load_dotenv()

database_url = os.getenv("DATABASE_URL")


query = "SELECT * FROM jugadores_stats;"

with psycopg2.connect(database_url, client_encoding='utf8') as conn:
    df = pd.read_sql_query(query, conn)
df

C:\Users\aabap\AppData\Local\Temp\ipykernel_50444\2202220803.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,id,nombre,partido,titular,suplente,goles,asist,posicion,gol_x_part,asist_x_part,contribucion_gol,pct_partidos_jugados,pct_titularidad
0,1,Jorge Rodriguez,25,17,8,0,0,Port,0.00,0.00,0.00,83.3,68.0
1,2,Javier Sanchez,27,14,13,0,0,Port,0.00,0.00,0.00,90.0,51.9
2,3,Manuel Hermoso,25,19,6,0,0,Cent,0.00,0.00,0.00,83.3,76.0
3,4,Martin Peiro,22,6,16,0,0,Cent,0.00,0.00,0.00,73.3,27.3
4,5,Miguel Gutierrez,29,28,1,7,2,Cent,0.24,0.07,0.31,96.7,96.6
5,6,Kosta Velazco,25,19,6,5,6,Lat,0.20,0.24,0.44,83.3,76.0
6,7,Jaime Cerezo,27,24,3,4,7,Lat,0.15,0.26,0.41,90.0,88.9
7,8,Alejandro Martinez,21,5,16,2,5,Lat,0.10,0.24,0.33,70.0,23.8
8,9,Pablo Leon,19,14,5,1,3,Lat,0.05,0.16,0.21,63.3,73.7
9,10,Hugo Mauro,25,11,14,1,0,Lat,0.04,0.00,0.04,83.3,44.0


In [22]:
goles_df = df[df['goles'] > 0].loc[df.index.repeat(df['goles'])].copy()
goles_df['tipo_evento'] = 'Gol'

asist_df = df[df['asist'] > 0].loc[df.index.repeat(df['asist'])].copy()
asist_df['tipo_evento'] = 'Asistencia'

df_eventos = pd.concat([goles_df, asist_df], ignore_index=True)
df_eventos = df_eventos[['tipo_evento', 'posicion', 'nombre', 'pct_titularidad']]

df_eventos.head()

,tipo_evento,posicion,nombre,pct_titularidad
0,Gol,Cent,Miguel Gutierrez,96.6
1,Gol,Cent,Miguel Gutierrez,96.6
2,Gol,Cent,Miguel Gutierrez,96.6
3,Gol,Cent,Miguel Gutierrez,96.6
4,Gol,Cent,Miguel Gutierrez,96.6


In [23]:
le_evento = LabelEncoder()
le_posicion = LabelEncoder()

X_eventos = le_evento.fit_transform(df_eventos['tipo_evento']).reshape(-1, 1)
y_posicion = le_posicion.fit_transform(df_eventos['posicion'])

# Modelo de atribución por Posición
modelo_atribucion = MultinomialNB()
modelo_atribucion.fit(X_eventos, y_posicion)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [24]:
def predecir_probabilidad_evento(tipo_evento='Gol'):
    idx_evento = le_evento.transform([tipo_evento]).reshape(-1, 1)
    probs = modelo_atribucion.predict_proba(idx_evento)[0]
    
    resultado = pd.DataFrame({
        'Posicion': le_posicion.classes_,
        'Probabilidad (%)': np.round(probs * 100, 2)
    }).sort_values(by='Probabilidad (%)', ascending=False)
    
    print(f"--- Distribución de Probabilidad cuando ocurre un evento de: {tipo_evento} ---")
    return resultado

predecir_probabilidad_evento('Gol')

--- Distribución de Probabilidad cuando ocurre un evento de: Gol ---


,Posicion,Probabilidad (%)
1,Del,43.82
3,Med,32.02
2,Lat,19.10
0,Cent,5.06


In [25]:
total_goles_equipo = df['goles'].sum()
total_asist_equipo = df['asist'].sum()

df_prob_jugador = df[['nombre', 'posicion', 'goles', 'asist']].copy()

df_prob_jugador['Prob_Gol_Jugador (%)'] = (df_prob_jugador['goles'] / total_goles_equipo * 100).round(2)
df_prob_jugador['Prob_Asist_Jugador (%)'] = (df_prob_jugador['asist'] / total_asist_equipo * 100).round(2)

print(df_prob_jugador.sort_values(by='Prob_Gol_Jugador (%)', ascending=False).head())

               nombre posicion  goles  asist  Prob_Gol_Jugador (%)  \
15  Alejandro Galindo      Del     30      7                 29.41   
17          Hugo Diaz      Del     19      8                 18.63   
13     Diego Baptista      Med     15     21                 14.71   
18     Sergio Velazco      Del      7      5                  6.86   
4    Miguel Gutierrez     Cent      7      2                  6.86   

    Prob_Asist_Jugador (%)  
15                    9.21  
17                   10.53  
13                   27.63  
18                    6.58  
4                     2.63  
